In [25]:
# Framework tools in the kernel are async wrappers — _make_tool_wrapper
# (tool_repl.py) wraps each tool in `async def _wrapper(**kwargs)` that
# dispatches through run_tool. The kernel's ZMQInteractiveShell enables
# autoawait, so top-level `await` works in cells. You MUST await every
# tool call or you get a coroutine object, not the result.
#
# IMPORTANT: the wrapper returns a TRANSPORT ENVELOPE:
#   {"status": "Success", "result": <the tool's actual return dict>, "_elapsed_s": N}
# NOT the tool's own return dict directly. So bundlejs["routes"] is []
# (wrapper level) while bundlejs["result"]["routes"] has the real data.
#
# The kernel namespace ships envelope helpers that auto-unwrap:
#   keys(r)     → every distinct key name, at any depth
#   pluck(r, K) → flat list of every value for key K, at any depth
#   find(r, K)  → like pluck but with optional value match + path info
#   grep(r, RX) → (path, line) tuples for every string matching the regex
# USE THESE instead of guessing where a field lives in the nesting.

bundlejs = await extract_js_routes(url="http://192.168.56.107:80", insecure=True)

# --- discovery: what keys exist in this envelope? ---
print("all keys:", keys(bundlejs))
print()

# pluck auto-unwraps the envelope (via _envelope_payload) and walks
# every depth, so you don't need to know it's under ["result"].
routes = pluck(bundlejs, "routes")
js_urls = pluck(bundlejs, "url")
smaps = pluck(bundlejs, "sourcemaps")

print("status:", bundlejs["status"])
print("routes:", routes)
print("sourcemaps:", smaps)
print("js bundle URLs:")
for u in js_urls:
    print(f"  {u}")

  [repl] Running payloads.js_recon.extract_js_routes
  [repl] Transport: brain_dispatch
  [repl] Args: {"url": "http://192.168.56.107:80", "insecure": true}
  [repl] Implementation: payloads.js_recon.extract_js_routes
all keys: ['elapsed_s', 'errors', 'full_urls', 'js_files', 'note', 'page', 'routes', 'scripts_found', 'secrets', 'size', 'skipped', 'sourcemaps', 'status', 'url']

status: Success
routes: [0, 0, ['/gruyere/', '/hackxor_intro.php', '/jquery.min.js']]
sourcemaps: [[]]
js bundle URLs:
  http://192.168.56.107:80
  http://192.168.56.107:80/animatedcollapse.js
  http://192.168.56.107:80/jquery.min.js
  http://192.168.56.107:80/jquery.min.js
  http://192.168.56.107:80/animatedcollapse.js


2026-10-12 05:49:29,261 [INFO] [TOOL_EXECUTE] Executing Tool ID: payloads.js_recon.extract_js_routes | Path: payloads.js_recon.extract_js_routes | Args: {'url': 'http://192.168.56.107:80', 'insecure': True} | Session: 0


In [31]:
# extract_js_routes discards the raw JS — we only get the envelope. Use
# pluck() to get the bundle URLs (auto-unwraps the ["result"] layer), then
# fetch the raw JS in-kernel so harvest() has actual text to decode.
#
# NOTE: kernel-side requests bypass the framework scope gate. This is your
# lane — the gate is an operator-armed guard for the secretary/agents, not
# a kernel sandbox.
import requests

bundle_urls = pluck(bundlejs, "url")
# filter to .js bundles (pluck also grabs the page URL and any other "url" key)
bundle_urls = [u for u in bundle_urls if u.endswith(".js")]

raw = ""
for burl in bundle_urls:
    try:
        r = requests.get(burl, verify=False, timeout=15,
                         headers={"User-Agent": "framework-jsrecon/1.0"})
        raw += "\n/* === %s === */\n" % burl
        raw += r.text
    except Exception as e:
        print(f"  fetch failed {burl}: {e}")

# also pull inline <script> bodies from the entry page
try:
    page_url = pluck(bundlejs, "url")  # first "url" is the page URL
    page_url = [u for u in pluck(bundlejs, "url") if not u.endswith(".js")][0]
    rp = requests.get(page_url, verify=False, timeout=15)
    import re as _re
    for m in _re.finditer(r"<script[^>]*>(.+?)</script>", rp.text, _re.S | _re.I):
        blob = m.group(1).strip()
        if blob and "src=" not in blob[:20]:
            raw += "\n/* === inline <script> === */\n" + blob
except Exception as e:
    print(f"  page inline fetch failed: {e}")

print(f"raw JS corpus: {len(raw)} chars from {len(bundle_urls)} bundles + inline")

display(Markdown(pprint(raw[:2000])))  # show first 2k chars of the raw JS corpus

raw JS corpus: 141099 chars from 4 bundles + inline
('\n'
 '/* === http://192.168.56.107:80/animatedcollapse.js === */\n'
 '//** Animated Collapsible DIV v2.0- (c) Dynamic Drive DHTML code library: '
 'http://www.dynamicdrive.com.\r\n'
 "//** May 24th, 08'- Script rewritten and updated to 2.0.\r\n"
 "//** June 4th, 08'- Version 2.01: Bug fix to work with jquery 1.2.6 (which "
 'changed the way attr() behaves).\r\n'
 "//** March 5th, 09'- Version 2.2, which adds the following:\r\n"
 '\t\t\t//1) ontoggle($, divobj, state) event that fires each time a DIV is '
 'expanded/collapsed, including when the page 1st loads\r\n'
 '\t\t\t//2) Ability to expand a DIV via a URL parameter string, ie: '
 'index.htm?expanddiv=jason or index.htm?expanddiv=jason,kelly\r\n'
 '\r\n'
 "//** March 9th, 09'- Version 2.2.1: Optimized ontoggle event handler "
 'slightly.\r\n'
 "//** July 3rd, 09'- Version 2.4, which adds the following:\r\n"
 '\t\t\t//1) You can now insert rel="expand[divid] | collapse[divid] | '

<IPython.core.display.Markdown object>

In [27]:
import base64, binascii, re
from urllib.parse import unquote
from codecs import decode as _cd

ROUTE = re.compile(r'''["'`](/[A-Za-z0-9_\-][A-Za-z0-9_$\-./%{}]{1,})["'`]''')
FARGS = re.compile(r'(?:fetch|open|axios(?:\.\w+)?)\s*\(\s*["\'`](/[^"\'`]{0,80})["\'`]')
STR_LIT = re.compile(r'''"(?:[^"\\\n]|\\.){6,}"|'(?:[^'\\\n]|\\.){6,}'|`(?:[^`\\]|\\.){6,}`''')
ESC  = re.compile(r'\\x[0-9a-fA-F]{2}|\\u[0-9a-fA-F]{4}')
# URL-safe base64 (- _ ) — b64d() handled them but the regex never captured them
B64C = re.compile(r'''["\']([A-Za-z0-9+/\-_]{12,}={0,2})["\']''')
HEXC = re.compile(r'''["\'](?:0x)?([0-9a-fA-F]{16,})["\']''')
PCTC = re.compile(r'((?:%[0-9A-Fa-f]{2}){4,})')
FRA  = re.compile(r'fromCharCode\(\s*([0-9xXa-fA-F,\s]+?)\s*\)', re.I)
CCA  = re.compile(r'\[\s*(?:\d+\s*,\s*){8,}\d+\s*\]')
# string concatenation — minifier/webpack favorite: "/api/" + "v1/" + "users"
CONCAT = re.compile(r'''["'`]([^"'`]{2,})["'`]\s*\+\s*["'`]([^"'`]{2,})["'`]''')
# eval("...") with raw encoded string decoded at runtime
EVAL_RE = re.compile(r'''eval\(\s*["']([^"']{8,})["']\s*\)''')

def _ok(s):  # printable + url-shaped gate, kills base64/hex false positives
    if not s or len(s) < 4: return False
    vis = sum(c.isprintable() or c in '\t\n' for c in s) / len(s)
    return vis > .85 and '/' in s

def b64d(s):
    s = re.sub(r'[^A-Za-z0-9+/=\-_]', '', s).rstrip('=')
    s = s + '=' * (-len(s) % 4)
    for cand in (s, s.replace('-', '+').replace('_', '/')):
        try: return base64.b64decode(cand).decode('utf-8', 'replace')
        except Exception: continue
    return ''

def hexd(s):
    s = re.sub(r'[^0-9a-fA-F]', '', s)
    if len(s) % 2: return ''
    try: return binascii.unhexlify(s).decode('utf-8', 'replace')
    except Exception: return ''

def _uq(s, n=3):
    for _ in range(n):
        u = unquote(s)
        if u == s: break
        s = u
    return s

def join_concat(text, passes=6):
    """Collapse 'a'+'b'+'c' chains into 'abc' so ROUTE can catch them."""
    for _ in range(passes):
        new = CONCAT.sub(lambda m: '"%s%s"' % (m.group(1), m.group(2)), text)
        if new == text: break
        text = new
    return text

def harvest(text):
    """Yield (kind, label, decoded_candidate) — regex capture only, no eval."""
    for m in STR_LIT.finditer(text):                     # minifier escapes
        lit = m.group(0)
        if ESC.search(lit):
            inner = lit[1:-1]  # strip quotes before decode
            try: yield 'esc', lit[:40], _cd(inner, 'unicode_escape')
            except Exception: pass
    for m in re.finditer(r'atob\(\s*["\']([A-Za-z0-9+/=\-_]+)["\']', text, re.I):
        d = b64d(m.group(1))
        if d and d not in text: yield 'atob', m.group(0)[:40], d
    for m in B64C.finditer(text):
        d = b64d(m.group(1))
        if _ok(d) and d not in text: yield 'b64', m.group(1)[:40], d
    for m in HEXC.finditer(text):
        d = hexd(m.group(1))
        if _ok(d): yield 'hex', m.group(1)[:40], d
    for m in PCTC.finditer(text):
        d = _uq(m.group(1))
        if '/' in d: yield 'pct', m.group(1)[:40], d
    for pat in (FRA, CCA):                               # charcode joins
        for m in pat.finditer(text):
            toks = re.findall(r'0[xX][0-9a-fA-F]+|\d+', m.group(1))
            try: d = ''.join(chr(int(t, 0)) for t in toks)
            except Exception: continue
            if _ok(d): yield 'cc', m.group(0)[:40], d
    for m in EVAL_RE.finditer(text):                     # eval("encoded") — try all decoders
        payload = m.group(1)
        for dec, name in ((b64d, 'eval_b64'), (hexd, 'eval_hex'), (_uq, 'eval_pct')):
            d = dec(payload)
            if _ok(d):
                yield name, m.group(0)[:40], d
ROUTE_BARE = re.compile(r'''["'`](/[A-Za-z0-9_\-][A-Za-z0-9_$\-./%{}]{1,})["'`]''')
def net(t):
    return set(ROUTE.findall(t)) | set(FARGS.findall(t))

In [28]:
def workbench(raw, rounds=4):
    # Pre-pass: collapse string concatenation so joined routes become
    # route-shaped before harvest runs. This catches the webpack/minifier
    # "/api/" + "v1/" + "users" pattern that no single-blob decoder sees.
    seed = join_concat(raw)
    corpus = [('seed', 'concat', seed)]
    seen, frontier = set(), corpus
    for _ in range(rounds):                              # nested encodings
        nxt = []
        for kind, label, text in frontier:
            for k, lb, cand in harvest(text):
                if (k, cand) in seen: continue
                seen.add((k, cand)); nxt.append((k, lb, cand))
        if not nxt: break
        corpus += nxt; frontier = nxt

    raw_routes, prov = net(raw), {}
    for kind, label, text in corpus:
        for r in net(text):
            prov.setdefault(r, (kind, label, text))
    new = {r: p for r, p in prov.items() if r not in raw_routes}

    print(f"{len(raw_routes)} raw routes | {len(new)} NEW from decoding\n")
    for r in sorted(new):
        kind, label, text = new[r]
        i = max(text.find(r), 0)
        ctx = re.sub(r'\s+', ' ', text[max(0, i-45):i+len(r)+45]).strip()
        print(f"[{kind:7}] {r}\n          via {label}  |  …{ctx}…")
    return {"new": dict((r, new[r][:2]) for r in new),
            "all": sorted(prov), "raw": sorted(raw_routes)}

In [29]:
# Run it. result["new"] = routes decoded from encoded blobs that weren't
# visible in the raw text. result["raw"] = routes the regex net caught
# directly. result["all"] = union of both.
result = workbench(raw)

print("\n--- summary ---")
print(f"raw routes:  {len(result['raw'])}")
print(f"total routes: {len(result['all'])}")
print(f"new (decoded-only): {len(result['new'])}")
if result['new']:
    print("\nnew routes to feed to run_ffuf:")
    for r in sorted(result['new']):
        print(f"  {r}")

0 raw routes | 0 NEW from decoding


--- summary ---
raw routes:  0
total routes: 0
new (decoded-only): 0


/tmp/ipykernel_177663/4058552659.py:60: DeprecationWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. 
  try: yield 'esc', lit[:40], _cd(inner, 'unicode_escape')
/tmp/ipykernel_177663/4058552659.py:60: DeprecationWarning: "\]" is an invalid escape sequence. Such sequences will not work in the future. 
  try: yield 'esc', lit[:40], _cd(inner, 'unicode_escape')
/tmp/ipykernel_177663/4058552659.py:60: DeprecationWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. 
  try: yield 'esc', lit[:40], _cd(inner, 'unicode_escape')
